# 05 — Corpus Guardian (full text) e filtro ICE

Secondo corpus giornalistico, complementare al NYT: la **Guardian Open Platform API** restituisce il **testo completo** (`bodyText`) degli articoli, che il NYT non rende disponibile (paywall/ToS). Chiave *Developer* gratuita, uso non commerciale: **1 chiamata/s, 500/giorno**.

**Disambiguazione ICE.** Il Guardian ha un tag dedicato, `us-news/ice-us-immigration-and-customs-enforcement`, ma è applicato sistematicamente solo dal 2025 (prima: 0-26 articoli/anno, verificato 2026-09-25). Il Guardian inoltre scrive la sigla come **"Ice"** (house style), indistinguibile da *ice* = ghiaccio: la ricerca testuale sulla sigla è esclusa. Il filtro è quindi in due fasi:

1. **Disambiguazione (regola, questo notebook)**: tag ICE **oppure** nome completo `"Immigration and Customs Enforcement"` presente nel `bodyText` (verificato in locale: la ricerca per frase dell'API non è esatta).
2. **Rilevanza (semantica, fase successiva)**: distinguere articoli *su* ICE da menzioni di passaggio. Riferimento = articoli Guardian taggati, **non** le keyword NYT (userle come filtro renderebbe circolare il confronto NYT vs Guardian).

Pipeline: §1 fetch (raw, mai filtrato) → §2 parsing → §3 filtro a regola → §4 validazione su campione.

## Setup

In [ ]:
import datetime
import json
import logging
import os
import re
import time
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import requests
from dotenv import load_dotenv

%matplotlib inline

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = BASE_DIR / "data" / "raw" / "guardian" / "search"
PROCESSED_DIR = BASE_DIR / "data" / "processed" / "guardian"
CHECKPOINT_PATH = RAW_DIR.parent / "checkpoint.json"
OUT_CANDIDATES = PROCESSED_DIR / "articles_candidates.parquet"
OUT_ICE_RULE = PROCESSED_DIR / "articles_ice_rule.parquet"

SEARCH_URL = "https://content.guardianapis.com/search"
ICE_TAG = "us-news/ice-us-immigration-and-customs-enforcement"
ICE_FULLNAME = "Immigration and Customs Enforcement"
START_YEAR = 2003
PAGE_SIZE = 200  # massimo consentito dall'API
MAX_RETRIES = 5
BACKOFF_BASE_SECONDS = 30

# Due query indipendenti: il tag da solo perde quasi tutto il pre-2025,
# la frase da sola perde articoli taggati che usano solo "Ice".
QUERIES = {
    "phrase": {"q": f'"{ICE_FULLNAME}"'},
    "tag": {"tag": ICE_TAG},
}

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", force=True)
log = logging.getLogger("guardian_pipeline")
print(f"BASE_DIR: {BASE_DIR}")

In [ ]:
def load_config() -> tuple[str, float]:
    load_dotenv(BASE_DIR / ".env")
    api_key = os.getenv("GUARDIAN_API_KEY")
    if not api_key:
        raise RuntimeError("GUARDIAN_API_KEY non trovata in .env")
    rate_limit_seconds = float(os.getenv("GUARDIAN_RATE_LIMIT_SECONDS", "1.1"))
    return api_key, rate_limit_seconds

api_key, rate_limit_seconds = load_config()
print(f"GUARDIAN_API_KEY caricata: {'si' if api_key else 'no'}")
print(f"Rate limit configurato: {rate_limit_seconds}s tra le richieste")

## 1. Raccolta dati — Guardian Content API

Unità di lavoro = **(query, anno)**: con `page-size=200` un anno intero sta in 1-6 pagine, per un totale di ~50 chiamate per l'intero 2003→oggi (contro il limite di 500/giorno). Il checkpoint viene salvato dopo ogni unità completata.

Il raw di ogni unità è la lista delle risposte-pagina così come arrivano dall'API, in `data/raw/guardian/search/{query}/{anno}.json`. **L'anno corrente non viene mai marcato completato**: a ogni riesecuzione viene riscaricato (il nuovo file è un superset del precedente), gli anni chiusi non vengono più toccati.

Campi richiesti: `headline, standfirst, byline, bodyText, wordcount` + tag `keyword`. Ordinamento `oldest` per una paginazione stabile.

In [ ]:
def unit_key(query: str, year: int) -> str:
    return f"{query}:{year}"

def load_checkpoint() -> dict:
    if CHECKPOINT_PATH.exists():
        return json.loads(CHECKPOINT_PATH.read_text(encoding="utf-8"))
    return {"completed_units": [], "failed_units": [], "last_run": None}

def save_checkpoint(checkpoint: dict) -> None:
    checkpoint["last_run"] = datetime.datetime.now(datetime.timezone.utc).isoformat()
    CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
    CHECKPOINT_PATH.write_text(json.dumps(checkpoint, indent=2), encoding="utf-8")

def raw_json_path(query: str, year: int) -> Path:
    return RAW_DIR / query / f"{year}.json"

def fetch_page(session: requests.Session, api_key: str, params: dict) -> dict:
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = session.get(SEARCH_URL, params={**params, "api-key": api_key}, timeout=30)
        except requests.exceptions.RequestException as exc:
            log.warning("Errore di rete (tentativo %d/%d): %s", attempt, MAX_RETRIES, exc)
            time.sleep(BACKOFF_BASE_SECONDS * attempt)
            continue
        if response.status_code == 200:
            return response.json()["response"]
        if response.status_code in (401, 403):
            raise RuntimeError(f"API key non valida o non autorizzata (HTTP {response.status_code})")
        log.warning("HTTP %d (tentativo %d/%d): %s", response.status_code, attempt, MAX_RETRIES, response.text[:200])
        time.sleep(BACKOFF_BASE_SECONDS * attempt)
    raise RuntimeError(f"Fetch fallito dopo {MAX_RETRIES} tentativi: {params}")

def fetch_unit(session, api_key, rate_limit_seconds, query: str, year: int) -> list[dict]:
    base = {
        **QUERIES[query],
        "from-date": f"{year}-01-01",
        "to-date": f"{year}-12-31",
        "page-size": PAGE_SIZE,
        "order-by": "oldest",
        "show-fields": "headline,standfirst,byline,bodyText,wordcount",
        "show-tags": "keyword",
    }
    pages, page, n_pages = [], 1, 1
    while page <= n_pages:
        resp = fetch_page(session, api_key, {**base, "page": page})
        pages.append(resp)
        n_pages = resp.get("pages", 0)
        page += 1
        time.sleep(rate_limit_seconds)
    return pages

def run_fetch(start_year: int = START_YEAR, end_year: int | None = None) -> None:
    api_key, rate_limit_seconds = load_config()
    end_year = end_year or datetime.date.today().year
    current_year = datetime.date.today().year
    checkpoint = load_checkpoint()
    completed, failed = set(checkpoint["completed_units"]), set(checkpoint["failed_units"])

    units = [(q, y) for q in QUERIES for y in range(start_year, end_year + 1) if unit_key(q, y) not in completed]
    if not units:
        log.info("Niente da scaricare: tutte le unità risultano già completate.")
        return
    log.info("Unità da scaricare: %d", len(units))
    session = requests.Session()

    for query, year in units:
        key = unit_key(query, year)
        try:
            pages = fetch_unit(session, api_key, rate_limit_seconds, query, year)
        except RuntimeError as exc:
            log.error("%s: %s", key, exc)
            failed.add(key)
            checkpoint["failed_units"] = sorted(failed)
            save_checkpoint(checkpoint)
            continue
        path = raw_json_path(query, year)
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(json.dumps(pages, ensure_ascii=False), encoding="utf-8")
        n = sum(len(p.get("results", [])) for p in pages)
        log.info("Salvato %s -> %s (%d articoli, %d pagine)", key, path.relative_to(BASE_DIR), n, len(pages))

        failed.discard(key)
        if year < current_year:
            completed.add(key)
        checkpoint["completed_units"] = sorted(completed)
        checkpoint["failed_units"] = sorted(failed)
        save_checkpoint(checkpoint)

    log.info("Fatto. Completate: %d, fallite: %d", len(completed), len(failed))

In [ ]:
run_fetch()

## 2. Parsing

Unione delle due query, deduplica per `id` (un articolo taggato che contiene anche la frase compare in entrambe). Per ogni articolo, oltre ai campi testuali, calcoliamo in locale sul `bodyText`:

- `n_fullname`: occorrenze del nome completo, **case-insensitive** e con plurale tollerato (`Immigrations?`) — la prima versione con match esatto scartava articoli genuini scritti "US immigration and customs enforcement (Ice)" o con il refuso "Immigrations and Customs Enforcement" (5 su 10 nel campione degli scartati);
- `n_acronym`: occorrenze di `ICE`/`Ice` **case-sensitive** come parola intera. Da sola non disambigua (vedi intro): ha senso solo come misura di densità dentro articoli già disambiguati dalla regola.

In [ ]:
# Case-insensitive e plurale tollerato: il Guardian scrive anche "US immigration and customs
# enforcement (Ice)" e "Immigrations and Customs Enforcement" (trovati nella validazione §4).
FULLNAME_RE = re.compile(r"\bImmigrations?\s+and\s+Customs\s+Enforcement\b", re.IGNORECASE)
ACRONYM_RE = re.compile(r"\b(?:ICE|Ice)\b")

def extract_article(doc: dict, query: str) -> dict:
    fields = doc.get("fields") or {}
    tags = [t["id"] for t in doc.get("tags") or []]
    body = fields.get("bodyText") or ""
    return {
        "id": doc["id"],
        "pub_date": doc.get("webPublicationDate"),
        "section_id": doc.get("sectionId"),
        "section_name": doc.get("sectionName"),
        "pillar": doc.get("pillarName"),
        "headline": fields.get("headline") or doc.get("webTitle"),
        "standfirst": fields.get("standfirst"),
        "byline": fields.get("byline"),
        "body_text": body,
        "word_count": int(fields["wordcount"]) if fields.get("wordcount") else None,
        "web_url": doc.get("webUrl"),
        "keyword_tags_json": json.dumps(tags),
        "has_ice_tag": ICE_TAG in tags,
        "n_fullname": len(FULLNAME_RE.findall(body)),
        "n_acronym": len(ACRONYM_RE.findall(body)),
        "query": query,
    }

def iter_raw_files():
    return sorted(RAW_DIR.glob("*/*.json"))

rows = []
for path in iter_raw_files():
    query = path.parent.name
    for page in json.loads(path.read_text(encoding="utf-8")):
        rows.extend(extract_article(doc, query) for doc in page.get("results", []))

raw_df = pd.DataFrame(rows)
via = raw_df.groupby("id")["query"].agg(lambda s: sorted(set(s)))
df = raw_df.drop_duplicates("id").drop(columns="query").set_index("id")
df["via_phrase"] = via.map(lambda q: "phrase" in q)
df["via_tag"] = via.map(lambda q: "tag" in q)
df = df.reset_index()
df["pub_date"] = pd.to_datetime(df["pub_date"], utc=True, format="ISO8601")
df["year"] = df["pub_date"].dt.year
df = df.sort_values("pub_date").reset_index(drop=True)

print(f"Righe raw: {len(raw_df):,}  ->  articoli unici: {len(df):,}  (duplicati tra query: {len(raw_df) - len(df):,})")
print(f"Periodo: {df['pub_date'].min().date()} -> {df['pub_date'].max().date()}")
print(f"body_text vuoto: {(df['body_text'].str.len() == 0).sum()}")

## 3. Filtro a regola (disambiguazione)

`is_ice_rule = has_ice_tag OR n_fullname >= 1`

La seconda condizione scarta i risultati della query per frase che **non** contengono davvero il nome completo nel testo (la ricerca dell'API è approssimata: nel test preliminare 4 articoli su 18 non lo contenevano).

`agency_mentions = n_fullname + n_acronym` è salvato come colonna ma **non** usato come soglia qui: separare "articolo su ICE" da "menzione di passaggio" è il compito della fase di rilevanza (§ successiva / modello).

In [ ]:
df["is_ice_rule"] = df["has_ice_tag"] | (df["n_fullname"] >= 1)
df["agency_mentions"] = df["n_fullname"] + df["n_acronym"]

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
df.to_parquet(OUT_CANDIDATES, index=False)
ice = df[df["is_ice_rule"]].reset_index(drop=True)
ice.to_parquet(OUT_ICE_RULE, index=False)

print(f"Candidati: {len(df):,}  -> is_ice_rule: {len(ice):,}  (scartati: {len(df) - len(ice):,})")
print(f"  di cui con tag ICE: {ice['has_ice_tag'].sum():,}   solo nome completo: {(~ice['has_ice_tag']).sum():,}")
print(f"Scritti: {OUT_CANDIDATES.relative_to(BASE_DIR)}, {OUT_ICE_RULE.relative_to(BASE_DIR)}")

In [ ]:
yearly = df.groupby("year").agg(
    candidati=("id", "size"),
    ice_rule=("is_ice_rule", "sum"),
    con_tag=("has_ice_tag", "sum"),
)
yearly["scartati"] = yearly["candidati"] - yearly["ice_rule"]
mentions = ice.groupby("year")["agency_mentions"].describe()[["50%", "mean"]].rename(columns={"50%": "menzioni_mediana", "mean": "menzioni_media"})
yearly = yearly.join(mentions.round(1))
yearly

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
yearly[["ice_rule", "con_tag"]].plot(kind="bar", ax=ax, color=["#4C72B0", "#DD8452"], width=0.8)
ax.set_title("Guardian — articoli ICE per anno (filtro a regola) e quota con tag ICE")
ax.set_xlabel("")
ax.set_ylabel("articoli")
ax.legend(["is_ice_rule (tag OR nome completo)", "con tag ICE"])
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ice["agency_mentions"].clip(upper=30).plot(kind="hist", bins=30, ax=ax, color="#4C72B0")
ax.set_title("Menzioni dell'agenzia per articolo (troncato a 30)")
ax.set_xlabel("n_fullname + n_acronym")
plt.tight_layout()
plt.show()
print((ice["agency_mentions"] <= 2).mean().round(3), "quota di articoli con <= 2 menzioni (candidati 'di passaggio')")

## 4. Validazione su campione

Due controlli, come per il filtro NYT:

- **Falsi positivi della regola**: campione casuale di `is_ice_rule`, stratificato per periodo (stessi 4 periodi del notebook 02).
- **Articoli scartati**: campione dei candidati esclusi — devono essere risultati della query per frase che non contengono il nome completo e non hanno il tag.

Per ogni articolo: titolo, sezione, menzioni e la frase del testo in cui compare il nome completo.

In [ ]:
PERIODS = [("2003-2016", 2003, 2016), ("2017-2020", 2017, 2020), ("2021-2024", 2021, 2024), ("2025-presente", 2025, 9999)]

def context_snippet(body: str, width: int = 110) -> str:
    m = FULLNAME_RE.search(body) or ACRONYM_RE.search(body)
    if not m:
        return body[:width * 2]
    return "…" + body[max(0, m.start() - width): m.end() + width].replace("\n", " ") + "…"

def show_sample(frame: pd.DataFrame, n: int, seed: int = 42) -> pd.DataFrame:
    s = frame.sample(min(n, len(frame)), random_state=seed).sort_values("pub_date")
    return pd.DataFrame({
        "data": s["pub_date"].dt.date,
        "sezione": s["section_id"],
        "tag": s["has_ice_tag"],
        "menzioni": s["agency_mentions"],
        "titolo": s["headline"].str.slice(0, 90),
        "contesto": s["body_text"].map(context_snippet),
    })

pd.set_option("display.max_colwidth", 260)
samples = []
for label, y0, y1 in PERIODS:
    part = ice[(ice["year"] >= y0) & (ice["year"] <= y1)]
    samples.append(show_sample(part, 5).assign(periodo=label))
validation_sample = pd.concat(samples)
validation_sample

In [ ]:
show_sample(df[~df["is_ice_rule"]], 10)

<!-- s5-relevance -->
## 5. Filtro di rilevanza — articolo *su* ICE vs menzione di passaggio

Il filtro a regola (§3) dice se l'agenzia è **citata**, non se l'articolo **parla** di ICE: il 34% ha ≤2 menzioni, e nella validazione §4 compaiono articoli su Yemen, nomine giudiziarie, redistricting. Una soglia fissa sulle menzioni è arbitraria (un pezzo lungo su ICE può usare poche volte la sigla), quindi: **gold set annotato a mano → classificatore interpretabile → score su tutto il corpus**.

**Definizione operativa (codebook)** — da citare nel report:

| Etichetta | Criterio |
|---|---|
| `y` — **su ICE** | L'agenzia, le sue azioni (arresti, raid, detenzioni, deportazioni), il suo personale, le sue strutture o le sue politiche sono il tema o *un* tema centrale dell'articolo. Include pezzi su un singolo caso di detenzione/deportazione in cui ICE è l'attore. |
| `n` — **di passaggio** | ICE compare come contesto, elenco, esempio o citazione, ma l'articolo tratta altro (politica estera, nomine, elezioni, cultura, sport…). |
| `p` — **parziale** | Caso dubbio: ICE ha un ruolo secondario ma non trascurabile (es. pezzo sulla politica migratoria generale con un paragrafo su ICE). **Escluso dal training** (`PARTIAL_AS = None`); la scelta di unirlo a `y` o `n` resta modificabile e va dichiarata. |

**Anti-circolarità**: le etichette e le feature vengono solo da articoli Guardian — nessuna keyword NYT — così il confronto NYT vs Guardian resta indipendente.

Le annotazioni manuali **non sono rigenerabili** (sono input umano, non output di pipeline): stanno in `data/annotations/`, non in `data/processed/`, e il file non viene mai ricampionato né sovrascritto.

Pipeline: §5.1 feature → §5.2 campione stratificato → §5.3 annotazione → §5.4 modello + valutazione → §5.5 applicazione.

<!-- s5-relevance -->
### 5.1 Feature

Il `body_text` del Guardian non ha a capo (niente paragrafi), quindi la posizione è misurata come **offset relativo** nel testo.

| Feature | Intuizione |
|---|---|
| `log_mentions`, `density` (menzioni / 1.000 parole) | quanto spazio occupa l'agenzia, normalizzato per la lunghezza |
| `in_headline`, `in_standfirst` | l'impaginazione giornalistica mette il tema principale in alto: segnale probabilmente più forte |
| `first_pos` (0 = inizio, 1 = fine), `in_lead` (prima menzione nelle prime 100 parole) | una menzione solo in fondo è tipica del passaggio |
| `spread` (distanza prima–ultima menzione / lunghezza) | menzioni sparse su tutto il pezzo = tema ricorrente; una sola isolata = passaggio |
| `has_ice_tag` | etichetta editoriale Guardian (sistematica solo dal 2025: il modello non deve dipenderne troppo) |
| `is_live`, `log_words`, sezione (`us-news` / `commentisfree` / `world` / altro) | live-blog lunghissimi e sezioni non-news hanno profili di menzione diversi |

`ICE`/`Ice` qui si conta insieme al nome completo: dentro articoli già disambiguati dalla regola (nome completo o tag presenti), la sigla dopo il primo "(Ice)" è l'agenzia — la regola 1 del progetto (mai filtrare *sulla* sigla) resta rispettata, perché il filtro d'ingresso è §3.

In [ ]:
import numpy as np
from IPython.core.error import StdinNotImplementedError
from IPython.display import clear_output, display

ANNOTATIONS_DIR = BASE_DIR / "data" / "annotations"
GOLD_PATH = ANNOTATIONS_DIR / "guardian_relevance_gold.csv"
OUT_ICE_RELEVANT = PROCESSED_DIR / "articles_ice_relevant.parquet"

TAG_RE = re.compile(r"<[^>]+>")
LEAD_WORDS = 100


def agency_spans(text: str) -> list[int]:
    return sorted(m.start() for rx in (FULLNAME_RE, ACRONYM_RE) for m in rx.finditer(text))


def relevance_features(frame: pd.DataFrame) -> pd.DataFrame:
    body = frame["body_text"].fillna("")
    headline = frame["headline"].fillna("")
    standfirst = frame["standfirst"].fillna("").str.replace(TAG_RE, " ", regex=True)
    words = frame["word_count"].fillna(body.str.split().str.len()).clip(lower=1)

    spans = body.map(agency_spans)
    length = body.str.len().clip(lower=1)
    first = spans.map(lambda s: s[0] if s else np.nan)
    last = spans.map(lambda s: s[-1] if s else np.nan)
    lead_chars = body.map(lambda b: len(" ".join(b.split()[:LEAD_WORDS])))
    in_text = lambda s: s.str.contains(FULLNAME_RE) | s.str.contains(ACRONYM_RE)

    section = frame["section_id"].where(frame["section_id"].isin(["us-news", "commentisfree", "world"]), "other")
    feats = pd.DataFrame({
        "log_mentions": np.log1p(frame["agency_mentions"]),
        "density": 1000 * frame["agency_mentions"] / words,
        "in_headline": in_text(headline).astype(int),
        "in_standfirst": in_text(standfirst).astype(int),
        "first_pos": (first / length).fillna(1.0),
        "in_lead": (first < lead_chars).astype(int),
        "spread": ((last - first) / length).fillna(0.0),
        "has_ice_tag": frame["has_ice_tag"].astype(int),
        "is_live": frame["id"].str.contains("/live/", regex=False).astype(int),
        "log_words": np.log1p(words),
    }, index=frame.index)
    return feats.join(pd.get_dummies(section, prefix="sec", dtype=int))


ice = pd.read_parquet(OUT_ICE_RULE)
ice["period"] = pd.cut(ice["year"], [2002, 2016, 2020, 2024, 9999], labels=[p[0] for p in PERIODS])
X_all = relevance_features(ice)
log.info("Feature di rilevanza calcolate: %d articoli x %d feature", *X_all.shape)
X_all.describe().T.round(2)

<!-- s5-relevance -->
### 5.2 Campione stratificato da annotare

Un campione casuale sarebbe ~70% 2025+ e quasi tutto chiaramente rilevante: inutile per imparare **il confine**. Stratifichiamo su **periodo × fascia di menzioni** (1 / 2 / 3-5 / 6+) = 16 celle, `PER_CELL` articoli ciascuna (tutti se la cella ne ha meno). L'ordine di annotazione è mescolato, così la stanchezza dell'annotatore non si correla con periodo o fascia.

Il campione si crea **una volta sola**: se `GOLD_PATH` esiste viene riletto, mai ricampionato (le etichette già date resterebbero orfane). Per allargarlo usare `extend_gold()`, che aggiunge solo articoli nuovi.

I pesi di campionamento (`sample_weight` = articoli nella cella / articoli campionati) servono a riportare le metriche alla distribuzione reale del corpus in §5.4.

In [ ]:
PER_CELL = 13
SAMPLE_SEED = 42
MENTION_BINS = [0, 1, 2, 5, np.inf]
MENTION_LABELS = ["1", "2", "3-5", "6+"]

ice["mention_bin"] = pd.cut(ice["agency_mentions"], MENTION_BINS, labels=MENTION_LABELS)
# 0 menzioni = articolo con solo il tag (testo senza nome né sigla): finisce nella fascia più bassa
ice["mention_bin"] = ice["mention_bin"].fillna("1")
cell_sizes = ice.groupby(["period", "mention_bin"], observed=False).size()


def draw_gold_sample(exclude_ids: set[str], per_cell: int, seed: int) -> pd.DataFrame:
    pool = ice[~ice["id"].isin(exclude_ids)]
    parts = [g.sample(min(per_cell, len(g)), random_state=seed) for _, g in pool.groupby(["period", "mention_bin"], observed=True)]
    s = pd.concat(parts).sample(frac=1, random_state=seed)
    return pd.DataFrame({
        "id": s["id"], "period": s["period"].astype(str), "mention_bin": s["mention_bin"].astype(str),
        "headline": s["headline"], "web_url": s["web_url"], "label": "", "note": "",
    })


def load_or_create_gold() -> pd.DataFrame:
    if GOLD_PATH.exists():
        gold = pd.read_csv(GOLD_PATH, dtype=str, keep_default_na=False)
        log.info("Gold set esistente riletto: %d articoli, %d annotati", len(gold), (gold["label"] != "").sum())
        return gold
    gold = draw_gold_sample(set(), PER_CELL, SAMPLE_SEED)
    ANNOTATIONS_DIR.mkdir(parents=True, exist_ok=True)
    gold.to_csv(GOLD_PATH, index=False)
    log.info("Nuovo gold set campionato: %d articoli -> %s", len(gold), GOLD_PATH.relative_to(BASE_DIR))
    return gold


def extend_gold(per_cell: int, seed: int = SAMPLE_SEED + 1) -> pd.DataFrame:
    gold = load_or_create_gold()
    extra = draw_gold_sample(set(gold["id"]), per_cell, seed)
    gold = pd.concat([gold, extra], ignore_index=True)
    gold.to_csv(GOLD_PATH, index=False)
    log.info("Gold set esteso di %d articoli (totale %d)", len(extra), len(gold))
    return gold


gold = load_or_create_gold()
pd.crosstab(gold["period"], gold["mention_bin"], margins=True).reindex(columns=MENTION_LABELS + ["All"])

<!-- s5-relevance -->
### 5.3 Annotazione

Cella interattiva: per ogni articolo mostra titolo, sottotitolo, sezione, attacco del pezzo e fino a 4 **finestre di contesto** attorno alle menzioni (distribuite lungo il testo, così si vede se ICE torna o compare una volta sola). Tasti: `y` su ICE · `n` di passaggio · `p` parziale · `s` salta · `q` esci. Nota libera opzionale dopo l'etichetta, separata da spazio (`n yemen`).

**Il file viene salvato dopo ogni etichetta**: si può interrompere (`q`, stop del kernel) e riprendere quando si vuole, ricomincia dal primo non annotato. In esecuzione non interattiva (nbconvert, *Run All* senza input) la cella si limita a stampare lo stato. In alternativa si può annotare direttamente il CSV: colonna `label` ∈ {`y`,`n`,`p`}.

Obiettivo: tutto il campione (~200). Tempo stimato 20-30 s ad articolo.

In [ ]:
VALID_LABELS = {"y", "n", "p"}
ice_by_id = ice.set_index("id")


def mention_windows(body: str, k: int = 4, width: int = 180) -> list[str]:
    spans = agency_spans(body)
    if not spans:
        return []
    picks = [spans[i] for i in sorted({round(j * (len(spans) - 1) / max(k - 1, 1)) for j in range(k)})]
    return [f"[{100 * p / max(len(body), 1):.0f}%] …{body[max(0, p - width): p + width]}…" for p in picks]


def show_article(row: pd.Series, done: int, total: int) -> None:
    a = ice_by_id.loc[row["id"]]
    standfirst = TAG_RE.sub(" ", a["standfirst"] or "").strip()
    print(f"[{done}/{total}]  {a['pub_date'].date()}  ·  {a['section_id']}  ·  {a['word_count']} parole  ·  "
          f"menzioni {a['agency_mentions']}  ·  tag ICE {'sì' if a['has_ice_tag'] else 'no'}")
    print(f"\n### {a['headline']}\n{standfirst}\n")
    print("ATTACCO:", " ".join(a["body_text"].split()[:70]), "…\n")
    print(f"MENZIONI ({len(agency_spans(a['body_text']))} totali, [posizione nel testo]):")
    for w in mention_windows(a["body_text"]):
        print("  •", w)
    print(f"\n{a['web_url']}")


def annotate(gold: pd.DataFrame) -> pd.DataFrame:
    todo = gold.index[gold["label"] == ""].tolist()
    total = len(gold)
    for i in todo:
        clear_output(wait=True)
        show_article(gold.loc[i], total - len(todo) + todo.index(i) + 1, total)
        while True:
            answer = input("\n[y] su ICE · [n] passaggio · [p] parziale · [s] salta · [q] esci > ").strip()
            key, _, note = answer.partition(" ")
            key = key.lower()
            if key in VALID_LABELS | {"s", "q"}:
                break
            print("Tasto non valido.")
        if key == "q":
            break
        if key == "s":
            continue
        gold.loc[i, ["label", "note"]] = [key, note]
        gold.to_csv(GOLD_PATH, index=False)
    clear_output(wait=True)
    return gold


try:
    gold = annotate(gold)
except (EOFError, StdinNotImplementedError):
    log.info("Esecuzione non interattiva: annotazione saltata")

print(f"Annotati {(gold['label'] != '').sum()}/{len(gold)}")
print(gold["label"].replace("", "(da fare)").value_counts().to_string())

<!-- s5-relevance -->
### 5.4 Modello e valutazione

Due modelli a confronto, entrambi **regressione logistica** (con ~200 esempi un modello più complesso sovradatta):

- **A — feature esplicite** (§5.1): interpretabile, i coefficienti dicono *cosa* rende un articolo "su ICE".
- **B — feature + embedding** (`all-MiniLM-L6-v2` su titolo + sottotitolo + prime ~250 parole): aggiunge il contenuto semantico. Equivale a imparare un centroide "su ICE" dai soli articoli Guardian annotati.

Valutazione con **5-fold stratificato**, previsioni out-of-fold → metriche **per periodo** (il modello deve funzionare anche nel 2003-2016, non solo dove ci sono tanti dati). Le metriche "corpus" usano `sample_weight` per riportare il campione stratificato alla distribuzione reale.

**Soglia**: la più bassa che garantisce precisione ≥ `TARGET_PRECISION` sulle previsioni out-of-fold (stessa scelta precisione > recall del filtro NYT). 

**Scelta del modello** — A è il default (più facile da giustificare). Si adotta B se vale almeno una delle due:

1. **qualità globale**: l'average precision pesata di B supera quella di A di ≥ `MIN_AP_GAIN_FOR_B`;
2. **uniformità nel tempo**: il recall *minimo tra i periodi* di B supera quello di A di ≥ `MIN_RECALL_GAIN_FOR_B`.

Il criterio 2 è stato aggiunto il 2026-09-30, **dopo** aver visto i risultati del primo run (scelta post hoc, da dichiarare nel report): con il solo criterio 1 veniva scelto A, che però recupera ~60-70% degli articoli genuini pre-2025 contro ~95% nel 2025+. Un filtro con recall diverso per periodo altera proprio i confronti diacronici per cui il subset viene costruito (il salto 2025+ risulterebbe gonfiato dal filtro), quindi l'uniformità del recall conta più di un'AP quasi identica. Il confronto completo, incluse le alternative scartate, è in §5.6.

**Interpretabilità: cosa mostra (e cosa non mostra) il grafico dei coefficienti.**

- Il modello **A** decide con 14 feature che hanno un nome e un significato (densità delle menzioni, ICE nel titolo, posizione della prima menzione…). La regressione logistica assegna a ciascuna un coefficiente, quindi la logica si legge direttamente: "se ICE è nel titolo lo score sale di tanto, se la prima menzione è in fondo scende di tanto". Si può spiegare a parole.
- Il modello **B** usa le stesse 14 feature più 384 dimensioni di embedding (la rappresentazione del testo prodotta da MiniLM). L'embedding cattura il contenuto semantico dell'articolo, ma le singole dimensioni non hanno un significato nominabile: la dimensione 217 non vuol dire "parla di detenzioni". B funziona meglio, ma di un singolo articolo si può dire solo che "il testo somiglia a quelli etichettati `y`", non *quale* caratteristica lo ha fatto tenere o scartare.

Il grafico qui sotto è calcolato **sempre sul modello A** e ha senso solo per A. Quando la regola sceglie B (come nel run del 2026-09-30), il grafico **non descrive il filtro che ha prodotto `articles_ice_relevant.parquet`**: va letto, e citato nel report, come indicazione qualitativa di quali segnali distinguono un articolo su ICE da una menzione di passaggio. La giustificazione del modello B non passa dai coefficienti ma dalle metriche in cross-validation (tabelle qui sotto) e dal controllo dei casi vicini alla soglia (§5.5).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, precision_recall_curve, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

MIN_LABELS = 80
PARTIAL_AS = None      # None = 'p' escluso dal training; 0 o 1 per unirlo a 'n' o 'y'
TARGET_PRECISION = 0.90
MIN_AP_GAIN_FOR_B = 0.03
# Il subset serve a confronti tra periodi: un recall disomogeneo nel tempo distorce il confronto più di un'AP leggermente inferiore
MIN_RECALL_GAIN_FOR_B = 0.10
EMBED_MODEL = "all-MiniLM-L6-v2"
EMBED_CACHE = PROCESSED_DIR / "relevance_embeddings_minilm.parquet"
CV_SEED = 42
for noisy in ("httpx", "sentence_transformers", "huggingface_hub"):
    logging.getLogger(noisy).setLevel(logging.WARNING)

X_all.index = ice["id"].values
labeled = gold[gold["label"].isin(VALID_LABELS)].copy()
corpus_cells = {(str(p), str(b)): n for (p, b), n in cell_sizes.items()}


def build_train(partial_as) -> pd.DataFrame:
    data = labeled.assign(y=labeled["label"].map({"y": 1, "n": 0, "p": partial_as}))
    data = data.dropna(subset=["y"]).astype({"y": int}).reset_index(drop=True)
    # Peso = articoli della cella nel corpus / articoli annotati della cella: riporta il campione stratificato al corpus reale
    labeled_cells = data.groupby(["period", "mention_bin"]).size()
    data["w"] = [corpus_cells[(p, b)] / labeled_cells[(p, b)] for p, b in zip(data["period"], data["mention_bin"])]
    return data


train = build_train(PARTIAL_AS)
ready = len(train) >= MIN_LABELS and train["y"].nunique() == 2


def embed_corpus() -> pd.DataFrame:
    # Cache su disco: ~3k testi su CPU richiedono qualche minuto; si ricalcola solo se cambia l'insieme di articoli
    if EMBED_CACHE.exists():
        cached = pd.read_parquet(EMBED_CACHE)
        if set(cached.index) == set(ice["id"]):
            return cached.loc[ice["id"]]
    from sentence_transformers import SentenceTransformer
    texts = (
        ice["headline"].fillna("") + ". "
        + ice["standfirst"].fillna("").str.replace(TAG_RE, " ", regex=True) + " "
        + ice["body_text"].fillna("").map(lambda b: " ".join(b.split()[:250]))
    ).tolist()
    log.info("Embedding di %d articoli con %s...", len(texts), EMBED_MODEL)
    vecs = SentenceTransformer(EMBED_MODEL).encode(texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
    emb = pd.DataFrame(vecs, index=ice["id"].values, columns=[f"e{i}" for i in range(vecs.shape[1])])
    emb.to_parquet(EMBED_CACHE)
    return emb


def make_model():
    return make_pipeline(StandardScaler(), LogisticRegression(C=0.5, class_weight="balanced", max_iter=5000))


def pick_threshold(y, proba, w) -> float:
    prec, rec, thr = precision_recall_curve(y, proba, sample_weight=w)
    ok = np.where(prec[:-1] >= TARGET_PRECISION)[0]
    if len(ok) == 0:
        log.warning("Nessuna soglia raggiunge precisione %.2f: uso quella a precisione massima", TARGET_PRECISION)
        return float(thr[np.argmax(prec[:-1])])
    return float(thr[ok[0]])


def evaluate(name: str, X: pd.DataFrame, data: pd.DataFrame) -> tuple[dict, pd.DataFrame]:
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=CV_SEED)
    proba = cross_val_predict(make_model(), X.values, data["y"].values, cv=cv, method="predict_proba")[:, 1]
    y, w = data["y"].values, data["w"].values
    thr = pick_threshold(y, proba, w)
    pred = (proba >= thr).astype(int)
    rows = []
    for period, idx in [("corpus (pesato)", data.index)] + list(data.groupby("period").groups.items()):
        yy, pp, ww = y[idx], pred[idx], w[idx]
        rows.append({
            "modello": name, "gruppo": period, "n": len(idx), "positivi": int(yy.sum()),
            "precision": precision_score(yy, pp, sample_weight=ww, zero_division=0),
            "recall": recall_score(yy, pp, sample_weight=ww, zero_division=0),
        })
    summary = {
        "modello": name, "soglia": thr,
        "ROC-AUC": roc_auc_score(y, proba, sample_weight=w),
        "avg precision": average_precision_score(y, proba, sample_weight=w),
    }
    return summary, pd.DataFrame(rows)


if not ready:
    log.warning("Annotazioni insufficienti per il modello: %d utilizzabili (servono >= %d, entrambe le classi). "
                "Completa §5.3 e riesegui da qui.", len(train), MIN_LABELS)
else:
    log.info("Training su %d articoli (%d su ICE, %d di passaggio; %d parziali esclusi)",
             len(train), train["y"].sum(), (train["y"] == 0).sum(), (labeled["label"] == "p").sum() if PARTIAL_AS is None else 0)

    emb_all = embed_corpus()
    XA = X_all.loc[train["id"]]
    XB = XA.join(emb_all)

    MODEL_A, MODEL_B = "A: feature", "B: feature + embedding"
    results = [evaluate(MODEL_A, XA, train), evaluate(MODEL_B, XB, train)]
    summaries = pd.DataFrame([r[0] for r in results]).set_index("modello")
    per_period = pd.concat([r[1] for r in results])

    ap_gain = summaries.loc[MODEL_B, "avg precision"] - summaries.loc[MODEL_A, "avg precision"]
    min_recall = per_period[per_period["gruppo"] != "corpus (pesato)"].groupby("modello")["recall"].min()
    recall_gain = min_recall[MODEL_B] - min_recall[MODEL_A]
    CHOSEN = MODEL_B if ap_gain >= MIN_AP_GAIN_FOR_B or recall_gain >= MIN_RECALL_GAIN_FOR_B else MODEL_A
    THRESHOLD = summaries.loc[CHOSEN, "soglia"]
    log.info("Guadagno di B su A — AP: %+.3f (soglia %.2f) · recall minimo tra periodi: %.3f -> %.3f = %+.3f (soglia %.2f)",
             ap_gain, MIN_AP_GAIN_FOR_B, min_recall[MODEL_A], min_recall[MODEL_B], recall_gain, MIN_RECALL_GAIN_FOR_B)
    log.info("Modello scelto: %s, soglia %.3f", CHOSEN, THRESHOLD)
    display(summaries.round(3))
    display(per_period.round(3).pivot(index="gruppo", columns="modello", values=["precision", "recall"]))

    coef_model = make_model().fit(XA.values, train["y"].values)
    coefs = pd.Series(coef_model[-1].coef_[0], index=XA.columns).sort_values()
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.barh(coefs.index, coefs.values, color=np.where(coefs.values > 0, "#2a78d6", "#eb6834"), height=0.6)
    ax.axvline(0, color="#999999", linewidth=0.8)
    # I coefficienti sono leggibili solo per A: se il filtro applicato è B il grafico resta come lettura qualitativa, e va detto
    role = "è il modello applicato al corpus" if CHOSEN == MODEL_A else "lettura qualitativa: il modello applicato al corpus è B"
    ax.set_title(f"Modello A — coefficienti standardizzati ({role})\n(> 0 spinge verso 'su ICE', < 0 verso 'di passaggio')", fontsize=10)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.show()

<!-- s5-relevance -->
### 5.5 Applicazione al corpus

Il modello scelto viene riaddestrato su tutto il gold set e applicato ai 3.292 articoli di `articles_ice_rule.parquet`, che resta intatto. Output `articles_ice_relevant.parquet` = tutti gli articoli, con:

- `relevance_score` (probabilità 0-1) e `is_ice_relevant` (score ≥ soglia) — decisione **sempre del modello**, anche per gli articoli annotati, così il criterio è uniforme su tutto il corpus;
- `gold_label` — l'etichetta manuale dove esiste, per trasparenza e controlli successivi.

Oltre al parquet: quota di articoli tenuti per periodo e per anno (grafico a due pannelli: conteggi assoluti sopra, **percentuale di articoli tenuti** sotto — la scala dei conteggi è dominata dal 2025-2026 e nasconde gli anni precedenti; il 2026 è parziale, fino alla data dell'ultimo fetch), e confronto tra `is_ice_relevant` e le etichette manuali (incluse le `p`, che il modello non ha visto in training). Attenzione: per le righe `y`/`n` questo confronto è **in-sample** (il modello finale è addestrato proprio su quegli articoli) e con il modello B, che ha ~400 feature su 175 esempi, l'accordo è perfetto per costruzione. Non è una stima di qualità: quella è la cross-validation di §5.4. L'unica riga informativa è `p`.

Controllo finale: gli articoli più vicini alla soglia (i casi più incerti), da scorrere per verificare che il confine abbia senso.

In [ ]:
if not ready:
    log.warning("§5.5 saltata: modello non addestrato (vedi §5.4)")
else:
    X_final = X_all if CHOSEN.startswith("A") else X_all.join(emb_all)
    final_model = make_model().fit(X_final.loc[train["id"]].values, train["y"].values)

    out = ice.drop(columns=["mention_bin"]).copy()
    out["relevance_score"] = final_model.predict_proba(X_final.loc[out["id"]].values)[:, 1]
    out["is_ice_relevant"] = out["relevance_score"] >= THRESHOLD
    out["gold_label"] = out["id"].map(gold.set_index("id")["label"]).replace("", pd.NA)
    out["period"] = out["period"].astype(str)
    out.to_parquet(OUT_ICE_RELEVANT, index=False)
    log.info("Scritto %s: %d articoli, %d rilevanti (modello %s, soglia %.3f)",
             OUT_ICE_RELEVANT.relative_to(BASE_DIR), len(out), out["is_ice_relevant"].sum(), CHOSEN, THRESHOLD)

    kept = out.groupby("period").agg(articoli=("id", "size"), rilevanti=("is_ice_relevant", "sum"))
    kept["% tenuti"] = (100 * kept["rilevanti"] / kept["articoli"]).round(1)
    display(kept)

    yearly_rel = out.groupby("year").agg(regola=("id", "size"), rilevanti=("is_ice_relevant", "sum"))
    yearly_rel["pct"] = 100 * yearly_rel["rilevanti"] / yearly_rel["regola"]
    # Sotto questa numerosità la quota di un singolo anno è rumore (1 articolo sposta la % di oltre 10 punti)
    MIN_YEAR_N = 10
    reliable = (yearly_rel["regola"] >= MIN_YEAR_N).values
    overall_pct = 100 * out["is_ice_relevant"].mean()

    # Due pannelli con asse x condiviso invece di un secondo asse y: conteggi e quote hanno scale diverse
    fig, (ax, ax_pct) = plt.subplots(2, 1, figsize=(11, 6.5), sharex=True, gridspec_kw={"height_ratios": [3, 2]})
    yearly_rel[["regola", "rilevanti"]].plot(kind="bar", ax=ax, color=["#b7d3f6", "#2a78d6"], width=0.8)
    ax.set_title("Guardian — articoli ICE per anno: filtro a regola vs filtro di rilevanza")
    ax.set_ylabel("articoli")
    ax.legend(["regola (citazione)", f"rilevanti (score ≥ {THRESHOLD:.3f})"], frameon=False)
    ax.spines[["top", "right"]].set_visible(False)

    x = np.arange(len(yearly_rel))
    ax_pct.bar(x, yearly_rel["pct"], width=0.6, color=np.where(reliable, "#2a78d6", "#c9ccd1"))
    ax_pct.axhline(overall_pct, color="#666666", linewidth=0.8, linestyle="--")
    ax_pct.text(0.01, overall_pct + 2, f"intero corpus {overall_pct:.0f}%", transform=ax_pct.get_yaxis_transform(),
                fontsize=8, color="#555555", va="bottom")
    for xi, pct, ok in zip(x, yearly_rel["pct"], reliable):
        if ok:
            ax_pct.text(xi, pct + 2, f"{pct:.0f}%", ha="center", va="bottom", fontsize=8, color="#333333",
                            bbox={"facecolor": "white", "edgecolor": "none", "pad": 0.8})
    ax_pct.set_title(f"Quota di articoli tenuti dal filtro di rilevanza (grigio: anni con meno di {MIN_YEAR_N} articoli, quota non affidabile)",
                     fontsize=10)
    ax_pct.set_ylabel("% rilevanti")
    ax_pct.set_ylim(0, 100)
    ax_pct.set_xlabel("")
    ax_pct.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.show()

    display(pd.crosstab(out["gold_label"].fillna("(non annotato)"), out["is_ice_relevant"], margins=True))

    borderline = out.iloc[(out["relevance_score"] - THRESHOLD).abs().argsort()[:12]]
    display(borderline[["pub_date", "section_id", "agency_mentions", "relevance_score", "gold_label", "headline"]]
            .assign(pub_date=lambda d: d["pub_date"].dt.date).round(3))

<!-- s5-relevance -->
### 5.6 Analisi di sensibilità — trattamento delle `p` e scelta del modello

Due scelte di §5.4 non sono ovvie e vanno giustificate con i numeri, non a priori:

- **Cosa fare delle etichette `p`** (parziale): escluderle (`PARTIAL_AS = None`, scelta adottata), unirle a `y` o unirle a `n`.
- **Quale modello**: A, A senza `has_ice_tag` (la feature che esiste quasi solo dal 2025, sospettata di causare lo squilibrio di recall tra periodi), B.

Per ognuna delle 9 combinazioni: stessa procedura di §5.4 (5-fold, stessa soglia a precisione ≥ `TARGET_PRECISION`, metriche pesate), recall per periodo, numero di articoli del corpus che il filtro terrebbe e quante delle 33 `p` verrebbero tenute (conteggio out-of-sample solo nelle righe "p escluse"; nelle altre le `p` sono nel training e il numero riflette l'etichetta imposta). **Nessun file viene scritto**: la cella è solo diagnostica.

Avvertenza di lettura: quando le `p` entrano nel training cambia anche la definizione di "positivo" su cui le metriche sono calcolate, quindi le righe con trattamenti diversi delle `p` non misurano esattamente la stessa cosa. Il confronto più pulito è tra modelli **a parità di trattamento**.

In [ ]:
def sensitivity() -> pd.DataFrame:
    feature_sets = {
        "A": X_all,
        "A senza has_ice_tag": X_all.drop(columns=["has_ice_tag"]),
        "B": X_all.join(emb_all),
    }
    partial_ids = labeled.loc[labeled["label"] == "p", "id"]
    rows = []
    for p_name, partial_as in [("p escluse", None), ("p = y", 1), ("p = n", 0)]:
        data = build_train(partial_as)
        for m_name, X in feature_sets.items():
            summary, by_period = evaluate(m_name, X.loc[data["id"]], data)
            by_period = by_period.set_index("gruppo")
            score = make_model().fit(X.loc[data["id"]].values, data["y"].values).predict_proba(X.values)[:, 1]
            kept = pd.Series(score >= summary["soglia"], index=X.index)
            row = {
                "trattamento p": p_name, "modello": m_name, "n training": len(data),
                "AP": summary["avg precision"],
                "precision": by_period.loc["corpus (pesato)", "precision"],
                "recall": by_period.loc["corpus (pesato)", "recall"],
            }
            for period, _, _ in PERIODS:
                row[f"recall {period}"] = by_period.loc[period, "recall"]
            row["recall min"] = min(row[f"recall {period}"] for period, _, _ in PERIODS)
            row["articoli tenuti"] = int(kept.sum())
            row["p tenute (su 33)"] = int(kept.loc[partial_ids].sum())
            rows.append(row)
    return pd.DataFrame(rows).set_index(["trattamento p", "modello"])


if not ready:
    log.warning("§5.6 saltata: modello non addestrato (vedi §5.4)")
else:
    display(sensitivity().round(2))
